# Intro to plotting with python

Here are some examples to help you get used to working with pandas dataframes and making plots with matplotlib.

Intro guides to these packages:
- [10 minute intro to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Matplotlib quick start guide](https://matplotlib.org/stable/users/explain/quick_start.html)

This notebook works with the outputs of the chorismate mutase MD analysis which will be called something like:
- `frame_rms.dat`
- `res_rmsf.dat`
- `probe_batch_rawdf.pkl`
- `probe_waters_per_frame.dat`
- `arpeggio_batch_rawdf.pkl`
- `arpeggio_waters_per_frame.dat`

Your versions of these files might also include active site/ligand labels in the file name (like A128/B256/C384). Make sure to check at the start of each section that the command to read in the data is using the actual file name your data is saved under.

## Import packages

Any time you start working with this/restart the python kernel if it crashes/etc, you need to run this set of commands

In [1]:
### import all the necessary packages ###
import os, sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tol_colors as tc

# 1. Plotting cpptraj output data

To plot data from cpptraj, we want it in a normal data table format (file ending with `.out` instead of `.agr`).

## 1a. RMSD plots

In [ ]:
# read the data from frame_rmsd.dat into a dataframe called rmsd_df
rmsd_df = pd.read_csv('frame_rms.dat',sep=r'\s+',index_col=0)

# show the dataframe
rmsd_df

You can extract specific rows and columns from a dataframe with df.loc[rownames,columnnames]. A colon gets all rows/columns.

You can plot data from a dataframe (including one you've got with loc) by doing df.plot(plotoptions...)

In [ ]:
# plot: RMSDs for the whole trimer and each chain individually
fig, ax = plt.subplots(figsize=(10, 5))
rmsd_df.loc[:,['All','Ch1All','Ch2All','Ch3All']].plot(ylabel='RMSD from X-ray crystal structure',ax=ax)
ax.set_xlim(left=0,right=20000)
ax.set_title('Chain 1 vs Chain 2 vs Chain 3')

There are lots of ways of making the exact same plot. The block below makes the exact same plot as above with only one line of code by including all the options in the plot function.

In [ ]:
# figsize, xlim and title options added in, axes option removed since we haven't prepared a figure and axes first
rmsd_df.loc[:,['All','Ch1All','Ch2All','Ch3All']].plot(ylabel='RMSD from X-ray crystal structure',figsize=(10,5),xlim=(0,20000),title='Chain 1 vs Chain 2 vs Chain 3')

In [ ]:
# plot: Side chain/main chain RMSDs for each chain
fig, ax = plt.subplots(figsize=(10, 5))
rmsd_df.loc[:,['Ch1SC','Ch1MC','Ch2SC','Ch2MC','Ch3SC','Ch3MC']].plot(ylabel='RMSD from X-ray crystal structure',ax=ax)
ax.set_xlim(left=0,right=20000)
ax.set_title('SC vs MC')

## 1b. RMSF plots

In [ ]:
# read the data from res_rmsf.dat into a dataframe called rmsf_df
rmsf_df = pd.read_csv('res_rmsf.dat',sep=r'\s+',index_col=0)

# show the dataframe
rmsf_df

In [ ]:
# plot: RMSFs for whole residues
fig, ax = plt.subplots(figsize=(10, 5))
rmsf_df.loc[:,['rmsfCh1','rmsfCh2','rmsfCh3']].plot(ylabel='RMSF',ax=ax)
ax.set_title('whole residues')

The following example shows how to change plot colours. Here I want to use the tolcolors medium_contrast color scheme (shown below) since the three pairs of dark/light colours work well with this data. I don't want the white and black so I define the variable "cset" by taking the list of colours from the loaded tc package and removing the first and last elements.

![Medium contrast color scheme](https://sronpersonalpages.nl/~pault/images/scheme_mediumcontrast_extended.png)

In [ ]:
# plot: side chain and main chain RMSFs separately
cset = tc.medium_contrast[1:-1]
fig, ax = plt.subplots(figsize=(10, 5))
rmsf_df.loc[:,['rmsfCh1SC','rmsfCh1MC','rmsfCh2SC','rmsfCh2MC','rmsfCh3SC','rmsfCh3MC']].plot(ylabel='RMSF',ax=ax,color=cset)
ax.set_title('SC/MC separately')

# 2. Plotting Probe active site contact data

In [ ]:
with open('all-20000-probe_batch_rawdf.pkl','rb') as fp:
    probe_raw_df = pickle.load(fp)
wats = [res for res in probe_raw_df.index.levels[0].values if res.endswith('WAT')]

len(set(wats))

In [ ]:
# read in data
with open('all-20000-probe_batch_rawdf.pkl','rb') as fp:
    probe_raw_df = pickle.load(fp)

# just in case there's any waters in here, remove them
wats = [res for res in probe_raw_df.index.levels[0].values if res.endswith('WAT')]
probe_raw_df = probe_raw_df.drop(index=wats,level=0)
probe_raw_df

We can simplify this dataframe because we only care about the total contact counts

In [ ]:
# extract just the total column to its own dataframe 
probe_df = probe_raw_df['p_tot']
probe_df.name = 'probe'

# dataframe currently is one very long column with two sets of row labels
# reshape it so that the first set of row labels (the functional groups) become the columns 
probe_df = probe_df.unstack(level=0,fill_value=0)
probe_df

We removed the waters from the full contact count dataframe above because they move freely in and out of the active site during the simulation so their contacts can be quite transient. For example across this whole simulation, a total of 38 water molecules have contacts with the A:128 substrate in at least one frame. But no more than 8 waters have contacts with the substrate at the same time/in the same frame. 

Instead of looking at individual water molecules, we just look at the number of waters that have contacts in each frame. This info is already calculated by the batch rin function so we can just read it in.

In [ ]:
# read in the table listing how many waters have contacts in each frame
wats = open('all-20000-probe_waters_per_frame.dat','r').readlines()[1:-4]
probewatperframe = {}
for w in wats:
    w = w.strip().split()
    probewatperframe[w[0]] = float(w[1])
probewatperframe = pd.DataFrame.from_dict(probewatperframe,orient='index',columns=['WAT'])
probewatperframe

## 2a. Plot number of frames with contacts and mean contact counts for each protein FG

In [ ]:
# calculate number of frames where each group has contacts and mean contact count
nf = probe_df.apply(np.count_nonzero)
nf.sort_values(ascending=False,inplace=True)
nf = pd.concat([nf,probe_df.mean()],axis=1)
nf.columns = ['n_frames','contact_mean']
nf

In [ ]:
# plot bar chart of these values with two y-axes for the different scales
nf.plot(kind='bar', secondary_y='contact_mean', rot=90, legend=0)
ax1, ax2 = plt.gcf().get_axes() # gets the current figure and then the axes
ax1.set_ylabel('number of frames with contacts',color=tc.vibrant.blue)
ax2.set_ylabel('mean contact count',color=tc.vibrant.magenta)

## 2b. Violin plots of contact count spreads

Violin plots (also sometimes called swarm plots) are like a combination of a box plot and histogram, and are good for showing spread/density of data values. They usually contain a line in the middle showing the median and quartiles (like a box plot) and then the outer shape shows the density of values (like a histogram).

Before we make the plots in this example we reorder the functional groups by their median value so we get a sort of downwards slope across the plot.

In [ ]:
# this makes a data column of the medians and sorts it in descending order
ranks = probe_df.median().sort_values(ascending=False)
# then we take the reordered list of functional groups (the index/row labels from the ranks dataframe) and apply it to the original dataframe to reorder it
probe_df = probe_df.loc[:,list(ranks.index.values)]

# make plot
plt.violinplot(probe_df,range(len(probe_df.columns.values)),points=1000,showmedians=True)
plt.xticks(ticks=range(len(probe_df.columns.values)),labels=probe_df.columns.values,rotation='vertical')
plt.ylim(bottom=0)
plt.ylabel('number of probe contacts')
plt.show()

## 2c. Plot no. of each type of FG with contacts in each frame

Just like how we've already got a dataframe of how many water molecules have contacts with the substrate in each frame, we can group together the protein functional groups by side chain/main chain to count how many of each of those have contacts in each frame.

In [ ]:
# transpose the dataframe so the functional groups are the rows and frames are columns
typedf = probe_df.T

# create a second level of row labels of functional group types
fg_types = [(f,'SC') if f.endswith('SC') else (f,'MC') if f.endswith('MC') else (f, 'WAT') for f in typedf.index.values]
typedf.index = pd.MultiIndex.from_tuples(fg_types)

# group together all rows of the same FG type to count how many of them have contacts in each particular frame
typedf = typedf.groupby(level=1).agg(np.count_nonzero)

# transpose it back so the frames are the rows and the functional group types are the columns and make SC before MC because I prefer that
typedf = typedf.T
typedf = typedf.loc[:,['SC','MC']]

# add in water counts as well
typedf = pd.concat([probewatperframe,typedf],axis=1)

# turn "2cht.00001, 2cht.00002, ..." row labels into just the integer values of the frame numbers
typedf.index = [int(i.split('.')[1]) for i in typedf.index.values]

typedf

In [ ]:
fig, ax = plt.subplots(figsize=(21, 6))
typedf.plot(ax=ax)
ax.set_xlim(left=0,right=20000)

# 3. Plotting Arpeggio active site data

In [ ]:
# read in data
with open('all-20000-arpeggio_batch_rawdf.pkl','rb') as fp:
    arpeggio_raw_df = pickle.load(fp)

arpeggio_raw_df

In [ ]:
# extract just the total contact counts and reshape like in the probe example
### process dataframe to make it easier to work with ###
arpeggio_df = arpeggio_raw_df['a_tot']
arpeggio_df.name = 'arpeggio'
arpeggio_df = arpeggio_df.unstack(level=0,fill_value=0)
arpeggio_df = arpeggio_df.loc[:,[c for c in arpeggio_df.columns.values if not c.endswith('WAT')]]

arpeggio_df

In [ ]:
# read in the table listing how many waters have contacts in each frame
wats = open('all-20000-arpeggio_waters_per_frame.dat','r').readlines()[1:-4]
arpeggiowatperframe = {}
for w in wats:
    w = w.strip().split()
    arpeggiowatperframe[w[0]] = float(w[1])
arpeggiowatperframe = pd.DataFrame.from_dict(arpeggiowatperframe,orient='index',columns=['WAT'])
arpeggiowatperframe

## 3a. Plot number of frames with contacts

This is the exact same as the probe example in 2a but with the arpeggio data

In [ ]:
# calculate number of frames where each group has contacts and mean contact count
nf = arpeggio_df.apply(np.count_nonzero)
nf.sort_values(ascending=False,inplace=True)
nf = pd.concat([nf,arpeggio_df.mean()],axis=1)
nf.columns = ['n_frames','contact_mean']

# plot bar chart of these values with two y-axes for the different scales
nf.plot(kind='bar', secondary_y='contact_mean', rot=90, legend=0)
ax1, ax2 = plt.gcf().get_axes() # gets the current figure and then the axes
ax1.set_ylabel('number of frames with contacts',color=tc.vibrant.blue)
ax2.set_ylabel('mean contact count',color=tc.vibrant.magenta)

## 3b. Histograms of contact count spreads

The range of contact count values you get from Arpeggio is much lower than for probe so you don't get nice smooth violin plots. Instead let's do histograms.

In [35]:
# first order the functional groups by mean so the plot looks nicer
# this makes a data column of the means and sorts it in descending order
ranks = arpeggio_df.mean().sort_values(ascending=False)
# then we take the reordered list of functional groups (the index/row labels from the ranks dataframe) and apply it to the original dataframe to reorder it
arpeggio_df = arpeggio_df.loc[:,list(ranks.index.values)]

The code block below replaces manually fussing with your subplot layout/x-axis limits/etc. First you pick how many subplots you want on each row (I've picked 7 but that can be changed to whatever), and it will divide the number of FGs by your chosen number to work out how many rows are needed to fit all the subplots. The plotting code below uses these values to determine how big the figure will be. The maximum contact count value is used to set the x-axis limits and also sets the binning of the histograms to have a bar for each integer value rather than lumping ranges together.

Here I picked spwidth to be 7 because a) I know that for the A/C active site (ligand A:128), there are 21 FGs identified by arpeggio and b) there are 7 colours in the tol-vibrant colour scheme that I've set as my default colours and that looks nice with the plots lining up.

In [ ]:
# now work out what size grid we need to fit a subplot for every FG
numfg = len(list(ranks.index.values))
# pick how many subplots we want on one row
spwidth = 7
# get quotient and remainder when you divide number of FGs by chosen width
q,r = np.divmod(numfg,spwidth)
# if the remainder is non-zero, we need one more row than the quotient
if r:
    q += 1

# also find maximum contact count value to work out x limits
maxval = arpeggio_df.values.max()

# print what the calculated values are so you know what to expect
print(f'figure will contain {numfg} subplots across {q} rows and {spwidth} columns. the maximum contact count value is {maxval}.')

In [ ]:
# make the plot
arpeggio_df.plot(kind='hist', # pick type of plot
                range=(0,maxval), # histogram range of values
                bins=maxval, # no. bins = max val so each val in own bin
                figsize=(2*spwidth,2*q), # set size so subplots approx 2x2
                subplots=True, # turn on subplots
                layout=(q,spwidth), # (rows,columns) of subplots
                title=list(arpeggio_df.columns), # title for each subplot
                legend=False, # turn off legend bc info in titles instead
                sharex=True, # share x labels
                sharey=True, # share y labels
                xlim=(0,maxval), # xlimits
                ylim=(0,20000)) # y limits
fig = plt.gcf()
# setting tight layout avoids stuff overlapping awkwardly
fig.tight_layout()
# set 
fig.supxlabel('Arpeggio contact count',y=-0.01)
fig.supylabel('Number of frames with contacts',x=-0.01)

## 3c. Plot no. of each type of FG with contacts

Again this is the exact same as the probe example just with the arpeggio data

In [ ]:
# transpose the dataframe so the functional groups are the rows and frames are columns
typedf = arpeggio_df.T

# create a second level of row labels of functional group types
fg_types = [(f,'SC') if f.endswith('SC') else (f,'MC') if f.endswith('MC') else (f, 'WAT') for f in typedf.index.values]
typedf.index = pd.MultiIndex.from_tuples(fg_types)

# group together all rows of the same FG type to count how many of them have contacts in each particular frame
typedf = typedf.groupby(level=1).agg(np.count_nonzero)

# transpose it back so the frames are the rows and the functional group types are the columns and make SC before MC because I prefer that
typedf = typedf.T
typedf = typedf.loc[:,['SC','MC']]

# add in water counts as well
typedf = pd.concat([arpeggiowatperframe,typedf],axis=1)

# turn "2cht.00001, 2cht.00002, ..." row labels into just the integer values of the frame numbers
typedf.index = [int(i.split('.')[1]) for i in typedf.index.values]

# make plot
fig, ax = plt.subplots(figsize=(21, 6))
typedf.plot(ax=ax)
ax.set_xlim(left=0,right=20000)

## 3d. Plot individual interaction types as stripes

At this point we have to go back to the original arpeggio dataframe to get the data for the individual contact types. The contact types that can be assigned by arpeggio are:
- steric clashes
- covalent bonds
- van der Waals clashes
- van der Waals interactions
- proximal interactions
- hydrogen bonds
- weak hydrogen bonds
- halogen bonds
- ionic interactions
- metal complexes
- aromatic interactions
- hydrophobic interactions
- carbonyl interactions
- polar contacts
- weak polar contacts

Not all of these are present in the chorismate mutase active site. The way we processed the data already removes proximal interactions (they are distance based and not really meaningful like the others) and there's no metal or halogen atoms present. Therefore we start by removing the columns that correspond to interaction types that are never present in this RIN. 

This example is much more complex than the others in this workbook. There's a lot of data manipulation involved in making the plots look how I want them to look. I've separated the code out into more blocks to show off the steps involved.

In [ ]:
# find types that are present at any point (have any non-zero values in the columns)
presenttypes = arpeggio_raw_df.columns[arpeggio_raw_df.any()].tolist()
# extract just those columns from the dataframe
arpeggio_types_df = arpeggio_raw_df.loc[:,presenttypes]

arpeggio_types_df

In [ ]:
# reorganise so that index is just frames and FGs are part of column labels
arpeggio_types_df = arpeggio_types_df.unstack(level=0,fill_value=0)
arpeggio_types_df = arpeggio_types_df.swaplevel(0,1,axis=1)
arpeggio_types_df = arpeggio_types_df.sort_index(level=0,axis=1)

# make the values just the frame numbers to make plotting easier
arpeggio_types_df.index = [int(i.split('.')[1]) for i in arpeggio_types_df.index.values]

arpeggio_types_df


In [ ]:
# turn into yes/no data: val is 1 if contact present, 0 if not present
arpeggio_types_df = arpeggio_types_df.astype('bool').astype('int')

arpeggio_types_df


In [ ]:
# get list of FGs
fgs = list(arpeggio_types_df.columns.levels[0])
# remove waters from this list
fgs = [f for f in fgs if not f.endswith('WAT')]

fgs


In [ ]:
# create dictionary that lists how many frames with contacts for each FG
nf = {f: sum(arpeggio_types_df.loc[:,(f,'a_tot')]) for f in fgs}

nf


In [ ]:
# use that dictionary of values to sort the list of FGs
fgs = sorted(fgs, key = lambda x: nf[x], reverse=True)

fgs

In [ ]:
# similar to example 3b: work out subplot layout rows/cols automatically

numfg = len(fgs)
# pick how many subplots we want on one row
spwidth = 3
# get quotient and remainder when you divide number of FGs by chosen width
q,r = np.divmod(numfg,spwidth)
# if the remainder is non-zero, we need one more row than the quotient
if r:
    q += 1

# print what the calculated values are so you know what to expect
print(f'figure will contain {numfg} subplots across {q} rows and {spwidth} columns.')

Right now for each FG we have columns of 1s and 0s for each interaction type. If we plot these as is, all the lines will overlap. To stripe them out I just multiply each interaction type column by a different integer so that then they'll be at different points on the y-axis. 

This approach means we need to do stuff to each functional group's subset of data individually before plotting it, instead of plotting straight from the dataframe. First we set up a grid of subplots, and then for each functional group we extract that set of data, prepare it, and then plot it into one of the subplots.

We also want to use the proper names of the interaction types, not the really short ones used to keep the dataframe columns narrow. Rather than manually setting this (which means the code won't translate properly to other systems with different types), we can automate updating the labels. Doing this processing first can help us work out how many interaction types we'll have in the plot, which determines which integers we want to multiply each column by to stripe them out.

In [ ]:
# create dictionary that defines proper name for each short name
typenames = {'a_cl': 'clash', 'a_cov': 'covalent', 
            'a_vdwcl': 'van der Waals clash', 'a_vdw': 'van der Waals', 
            'a_hb': 'hydrogen bond', 'a_whb': 'weak hydrogen bond', 
            'a_hal': 'halogen', 'a_ion': 'ionic', 'a_met': 'metal', 
            'a_arom': 'aromatic', 'a_hp': 'hydrophobic', 'a_CO': 'carbonyl', 'a_polar': 'polar', 'a_wpol': 'weak polar'}

# get number of types present again but now also remove totals
presenttypes = arpeggio_raw_df.columns[arpeggio_raw_df.any()].tolist()
presenttypes.remove('a_tot')

# count how many types are present
numtypes = len(presenttypes)

# we want to multiply the columns by decreasing integers to maintain the order of the interaction types, starting at the top of the plot
# easiest way to do get the list of values to multiply by is to get the integers from 1 up to the number of types present, then reverse it
multlist = list(reversed(range(1,numtypes+1)))

# get type labels for present types from full dictionary
typelabels = [typenames[t] for t in presenttypes]
typelabels

An example of what we're going to do to each functional group's data before plotting it (shown for the A:108:TYR_SC functional group):

In [ ]:
arpeggio_types_df.loc[:,'A:108:TYR_SC'].loc[:,presenttypes].replace(0,np.nan) * multlist

If we plot this new dataframe, a point will be drawn at y=8 for each frame where the A:108:TYR_SC group has any van der waals clashes with the substrate, a point will be drawn at y=6 for each frame where there are hydrogen bond interactions found between A:108:TYR_SC and the substrate, etc etc. We make the full plot of all the functional groups together below by looping through the list `fgs`, first doing that setup and then plotting the data. The for loop/if condition section after the data has been processed just checks which subplot location needs to be used before plotting that set of data.

Also, it's SUPER IMPORTANT that we change the 0 values to NaNs (not a number = empty value) so that the line is not plotted when the contact type is not present. Values of 0 will be plotted normally, resulting in lines that go up and down instead of starting and stopping. You can try removing the `.replace(0,np.nan)` bit at the end of the eighth line of the code block below to see exactly why we don't want to plot values of 0.

In [ ]:
# set up plot
fig, axs = plt.subplots(nrows=q,ncols=spwidth,figsize=(3*spwidth, 2*q))
fig.tight_layout()

# now do the processing and plotting for each functional group
for i in range(len(fgs)):
    # extract required set of data, replace 0s with NaNs so not plotted
    pltdf = arpeggio_types_df.loc[:,fgs[i]].loc[:,presenttypes].replace(0,np.nan)
    # multiply by the list of integer values to stripe the data out
    pltdf = pltdf * multlist
    # check each row to see if plot belongs there
    for j in range(0,q):
        if j*spwidth <= i and i < (j+1)*spwidth:
            # if it does, plot it. x = frame, y = data we just picked out
            axs[j,i-j*spwidth].plot(list(pltdf.index),pltdf,linewidth=0.5)
            # put functional group name as subplot title
            axs[j,i-j*spwidth].set_title(fgs[i])
            # set x and y lims
            axs[j,i-j*spwidth].set_ylim(bottom=0,top=numtypes+1)
            axs[j,i-j*spwidth].set_xlim(0,20000)
            # only put y-axis labels on the leftmost plot of each row
            if i == j*spwidth:
                axs[j,i-j*spwidth].set_yticks(ticks=multlist,labels=typelabels)
            else:
                axs[j,i-j*spwidth].set_yticks(ticks=multlist,labels=[])

fig.supxlabel('frame',y=-0.01)

plt.show()